# Extração de features por CPE

Neste notebook é construído o conjunto principal de features utilizado no clustering dos CPEs.

O Set A é composto por 24 features homogéneas, correspondentes à percentagem média do consumo diário que ocorre em cada hora do dia. Todas as features estão na mesma escala (%), são diretamente interpretáveis e não requerem normalização adicional.

A análise de clustering será realizada com este conjunto de features, por representar de forma simples e comparável o perfil horário típico de cada CPE.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Permite importar módulos internos de src/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from config.paths import (
    RESULTS_DIR,
    RESULTS_EXPLORATION_DIR,
    RESULTS_FEATURES_DIR,
    RESULTS_CLUSTERING_DIR,
    RESULTS_ANOMALIES_DIR,
    RESULTS_MODELS_DIR,
)

# Pastas principais
base_dir = RESULTS_DIR
exploration_dir = RESULTS_EXPLORATION_DIR
features_dir = RESULTS_FEATURES_DIR
clustering_dir = RESULTS_CLUSTERING_DIR
anomalies_dir = RESULTS_ANOMALIES_DIR
models_dir = RESULTS_MODELS_DIR

# Subpastas específicas
intermediate_dir = exploration_dir / "intermediate"

# Clustering
clustering_analysis_dir = clustering_dir / "analysis"
clustering_plots_dir = clustering_dir / "plots"

# Features
features_plots_dir = features_dir / "plots"
features_validation_dir = features_dir / "validation"

# Anomalies
anomalies_plots_dir = anomalies_dir / "plots"
anomalies_temporal_plots_dir = anomalies_dir / "anomalies_temporal_plots"

for d in [
    exploration_dir,
    features_dir,
    clustering_dir,
    anomalies_dir,
    models_dir,
    intermediate_dir,
    clustering_analysis_dir,
    clustering_plots_dir,
    features_plots_dir,
    features_validation_dir,
    anomalies_plots_dir,
    anomalies_temporal_plots_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# Carregar outputs intermédios da exploração

df_pivot = pd.read_csv(intermediate_dir / "df_pivot.csv", index_col=0, parse_dates=True)
perfil_horario = pd.read_csv(intermediate_dir / "perfil_horario.csv", index_col=0)

perfil_horario.index = perfil_horario.index.astype(int)

print("Ficheiros intermédios carregados com sucesso.")
display(perfil_horario.head())

In [ ]:
# SET A — Perfil horário normalizado (24 features)

# Cada feature representa a percentagem do consumo diário total que ocorre numa determinada hora do dia.
# Exemplo: se f10 = 12.5%, significa que em média 12.5% do consumo diário deste CPE acontece entre as 10:00 e as 11:00.

# Normalizar: cada hora como % do total diário
perfil_horario_pct = perfil_horario.copy()
totais_diarios = perfil_horario_pct.sum(axis=0)  # soma das 24 horas por CPE
 
# Evitar divisão por zero (CPEs com consumo total ~0)
totais_diarios = totais_diarios.replace(0, np.nan)
 
perfil_horario_pct = (perfil_horario_pct / totais_diarios) * 100
 
# Transpor: cada linha = 1 CPE, cada coluna = 1 hora
features_setA = perfil_horario_pct.T.copy()
features_setA.columns = [f"f{h+1:02d}_pct_hora_{h:02d}" for h in range(24)]
features_setA.index.name = "CPE"

In [ ]:
# Verificação

print("SET A — Perfil horário normalizado (24 features)")

print(f"Shape: {features_setA.shape}")
print(f"Soma por CPE (deve ser ~100%):")
somas = features_setA.sum(axis=1)
print(f"  Min: {somas.min():.2f}%")
print(f"  Max: {somas.max():.2f}%")
print(f"  Mean: {somas.mean():.2f}%")

In [ ]:
# Remover CPEs com consumo total ~0 (se existirem)

mask_valido = ~features_setA.isna().any(axis=1)
n_removidos = (~mask_valido).sum()
if n_removidos > 0:
    print(f"\nRemovidos {n_removidos} CPEs com consumo total ~0")
    features_setA = features_setA[mask_valido]
 
print(f"\nShape final: {features_setA.shape}")
display(features_setA.head())

In [ ]:
# Visualização: perfil médio global

plt.figure(figsize=(10, 4))
features_setA.mean().plot(kind="bar", color="#0D7C66", alpha=0.8)
plt.title("Perfil médio horário (% do consumo diário)")
plt.xlabel("Hora do dia")
plt.ylabel("% do consumo diário")
plt.xticks(range(24), [f"{h}h" for h in range(24)], rotation=45)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(features_plots_dir / "perfil_horario_pct_medio.png", dpi=150)
plt.show()

In [ ]:
# Visualização: heatmap de todos os CPEs

plt.figure(figsize=(14, 8))
sns.heatmap(
    features_setA.sort_values("f01_pct_hora_00", ascending=False),
    cmap="YlOrRd",
    xticklabels=[f"{h}h" for h in range(24)],
    yticklabels=False,
    cbar_kws={"label": "% consumo diário"}
)
plt.title("Distribuição horária do consumo por CPE (Set A)")
plt.xlabel("Hora do dia")
plt.ylabel("CPE (ordenados por consumo nocturno)")
plt.tight_layout()
plt.savefig(features_plots_dir / "heatmap_perfil_horario_pct.png", dpi=150)
plt.show()

In [ ]:
# Guardar features para clustering

features_cpe = features_setA.copy()

features_cpe.to_csv(features_dir / "features_cpe.csv")
features_cpe.to_csv(features_dir / "features_setA.csv")

print(f"\nFeatures finais guardadas em: {features_dir / 'features_cpe.csv'}")
print(f"Cópia de compatibilidade guardada em: {features_dir / 'features_setA.csv'}")

In [ ]:
# Estatísticas descritivas

print("\nEstatísticas descritivas do Set A:")
display(features_setA.describe().T.round(2))

In [ ]:
# Resumo

print("RESUMO")
print(f"\n  Features finais: {features_cpe.shape[1]} features · {features_cpe.shape[0]} CPEs")
print("\n  → O clustering será realizado com estas features.")
print("  → As features representam a percentagem média de consumo diário por hora.")
print("  → Todas estão na mesma escala (%), pelo que não exigem normalização adicional.")

# Validação e análise das features extraídas

Antes de avançar para o clustering, é importante validar que as features são informativas e não redundantes.

In [ ]:
# Correlação entre features - Set A

plt.figure(figsize=(14, 10))

corr_A = features_setA.corr()
mask = np.triu(np.ones_like(corr_A, dtype=bool))

sns.heatmap(
    corr_A,
    mask=mask,
    annot=True,
    fmt=".1f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 7}
)

plt.title("Matriz de correlação das features — Set A")
plt.tight_layout()
plt.savefig(features_validation_dir / "correlacao_features_setA.png", dpi=150)
plt.show()

# Identificar pares altamente correlacionados (|r| > 0.9)
high_corr_pairs_A = []

for i in range(len(corr_A.columns)):
    for j in range(i + 1, len(corr_A.columns)):
        if abs(corr_A.iloc[i, j]) > 0.9:
            high_corr_pairs_A.append({
                "feature_1": corr_A.columns[i],
                "feature_2": corr_A.columns[j],
                "correlacao": corr_A.iloc[i, j]
            })

if high_corr_pairs_A:
    print("Set A — Pares de features com correlação > 0.9:")
    display(
        pd.DataFrame(high_corr_pairs_A).sort_values(
            "correlacao", key=abs, ascending=False
        )
    )
    print("\nNota: features altamente correlacionadas podem ser redundantes.")
    print("Considerar remoção de uma de cada par antes do clustering.")
else:
    print("Set A — Nenhum par de features com correlação > 0.9 encontrado.")

In [ ]:
# Distribuição das features - Set A

fig, axes = plt.subplots(5, 5, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(features_setA.columns):
    if i < len(axes):
        features_setA[col].hist(
            bins=20,
            ax=axes[i],
            color="#0D7C66",
            alpha=0.7,
            edgecolor="white"
        )
        axes[i].set_title(col, fontsize=9, fontweight="bold")
        axes[i].tick_params(labelsize=7)

# Esconder eixos não usados
for j in range(len(features_setA.columns), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuição das features por CPE — Set A", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(features_validation_dir / "distribuicao_features_setA.png", dpi=150)
plt.show()

In [ ]:
# Estatísticas descritivas completas - Set A

print("\nSet A — Estatísticas descritivas:")
display(features_setA.describe().T.round(3))